## CodePath AI — Data Audit

In [24]:
import pandas as pd

In [25]:
df = pd.read_csv("../../data/raw/stackoverflow_full.csv")

In [26]:
len(df.columns)

15

In [27]:
len(df)

73462

In [28]:
df.columns

Index(['Unnamed: 0', 'Age', 'Accessibility', 'EdLevel', 'Employment', 'Gender',
       'MentalHealth', 'MainBranch', 'YearsCode', 'YearsCodePro', 'Country',
       'PreviousSalary', 'HaveWorkedWith', 'ComputerSkills', 'Employed'],
      dtype='str')

In [29]:
df.dtypes

Unnamed: 0          int64
Age                   str
Accessibility         str
EdLevel               str
Employment          int64
Gender                str
MentalHealth          str
MainBranch            str
YearsCode           int64
YearsCodePro        int64
Country               str
PreviousSalary    float64
HaveWorkedWith        str
ComputerSkills      int64
Employed            int64
dtype: object

In [30]:
df.head()

,Unnamed: 0,Age,Accessibility,EdLevel,Employment,Gender,MentalHealth,MainBranch,YearsCode,YearsCodePro,Country,PreviousSalary,HaveWorkedWith,ComputerSkills,Employed
0,0,<35,No,Master,1,Man,No,Dev,7,4,Sweden,51552.0,C++;Python;Git;PostgreSQL,4,0
1,1,<35,No,Undergraduate,1,Man,No,Dev,12,5,Spain,46482.0,Bash/Shell;HTML/CSS;JavaScript;Node.js;SQL;Typ...,12,1
2,2,<35,No,Master,1,Man,No,Dev,15,6,Germany,77290.0,C;C++;Java;Perl;Ruby;Git;Ruby on Rails,7,0
3,3,<35,No,Undergraduate,1,Man,No,Dev,9,6,Canada,46135.0,Bash/Shell;HTML/CSS;JavaScript;PHP;Ruby;SQL;Gi...,13,0
4,4,>35,No,PhD,0,Man,No,NotDev,40,30,Singapore,160932.0,C++;Python,2,0


In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73462 entries, 0 to 73461
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      73462 non-null  int64  
 1   Age             73462 non-null  str    
 2   Accessibility   73462 non-null  str    
 3   EdLevel         73462 non-null  str    
 4   Employment      73462 non-null  int64  
 5   Gender          73462 non-null  str    
 6   MentalHealth    73462 non-null  str    
 7   MainBranch      73462 non-null  str    
 8   YearsCode       73462 non-null  int64  
 9   YearsCodePro    73462 non-null  int64  
 10  Country         73462 non-null  str    
 11  PreviousSalary  73462 non-null  float64
 12  HaveWorkedWith  73399 non-null  str    
 13  ComputerSkills  73462 non-null  int64  
 14  Employed        73462 non-null  int64  
dtypes: float64(1), int64(6), str(8)
memory usage: 18.6 MB


### Initial Dataset Overview

Each row represents an individual developer or job applicant profile. The dataset contains 73,462 observations and 15 columns.

The primary numerical variables are YearsCode, YearsCodePro, PreviousSalary, and ComputerSkills. Although Employment and Employed are stored as integers, they represent binary categories rather than continuous numerical measurements.

The categorical variables include Age, Accessibility, EdLevel, Gender, MentalHealth, MainBranch, and Country.

HaveWorkedWith requires special treatment because it contains multiple semicolon-separated technologies in a single cell. It is therefore a multi-label feature rather than a standard categorical variable. Each technology will need to be separated and converted into an individual feature during preprocessing.

For the planned machine learning tasks:

PreviousSalary will be the target variable for the salary prediction model.

ComputerSkills will be the target variable for the skill benchmark model.

HaveWorkedWith will be used to construct developer skill vectors and identify similar profiles.

Unnamed: 0 appears to be an exported row index rather than a meaningful feature.

The presence of both Employment and Employed requires further investigation because their names suggest similar meanings. They should not be used in a model until their definitions and relationship have been verified.

## Data Quality Audit

In [32]:
df.isna().sum().sort_values(ascending=False)

HaveWorkedWith    63
Unnamed: 0         0
Age                0
Accessibility      0
EdLevel            0
Employment         0
Gender             0
MentalHealth       0
MainBranch         0
YearsCode          0
YearsCodePro       0
Country            0
PreviousSalary     0
ComputerSkills     0
Employed           0
dtype: int64

In [33]:
df.duplicated().sum()

np.int64(0)

In [34]:
df.duplicated(subset=([col for col in df.columns if col != 'Unnamed: 0'])).sum()

np.int64(4)

In [35]:
df.nunique()

Unnamed: 0        73462
Age                   2
Accessibility         2
EdLevel               5
Employment            2
Gender                3
MentalHealth          2
MainBranch            2
YearsCode            51
YearsCodePro         51
Country             172
PreviousSalary    12062
HaveWorkedWith    69980
ComputerSkills       73
Employed              2
dtype: int64

### Data Quality Assessment

Only the HaveWorkedWith column contains missing values. It has 63 missing observations, representing approximately 0.086% of the dataset. All 63 of these rows have a ComputerSkills value of zero, suggesting that no technologies were recorded for those profiles. This is a very small proportion of the dataset, but the rows should not be removed automatically before determining whether the missing value means “no reported skills” or “unknown information.”

No duplicate rows are detected when all columns are included. However, Unnamed: 0 contains a unique value for every row and therefore hides otherwise identical observations. When this index-like column is excluded, four duplicate observations are detected. These records should be investigated and deduplicated during preprocessing.

Several columns have low cardinality:

Age, Accessibility, Employment, MentalHealth, MainBranch, and Employed each contain two unique values.

Gender contains three unique values.

EdLevel contains five unique values.

In contrast, Country contains 172 unique values and HaveWorkedWith contains 69,980 unique technology combinations. Treating each complete HaveWorkedWith string as a separate category would create an extremely sparse and ineffective representation. The strings should instead be split by semicolons so that each individual technology becomes a separate binary feature.

Unnamed: 0 contains 73,462 unique values, matching the number of rows. This confirms that it is an identifier generated during CSV export and should be removed before model training.

No rows should be deleted during the audit stage. The current findings suggest that preprocessing will later need to:

Remove the Unnamed: 0 index column.

Remove duplicate records after excluding the index column.

Handle missing technology lists explicitly.

Split HaveWorkedWith into individual technologies.

Investigate Employment and Employed before deciding whether either should be retained.

## Numerical Feature Audit

In [36]:
numerical_columns = ["YearsCode", "YearsCodePro", "PreviousSalary", "ComputerSkills"]
numerical_summary = df[numerical_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
numerical_summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
YearsCode,73462.0,14.218902,9.405172,0.0,2.0,4.0,7.0,12.0,20.0,35.00,42.00,50.0
YearsCodePro,73462.0,9.098377,7.960201,0.0,0.0,1.0,3.0,7.0,12.0,25.00,36.00,50.0
PreviousSalary,73462.0,67750.260611,49488.142118,1.0,1788.0,6480.0,28839.0,57588.0,95979.0,168266.15,203308.53,224000.0
ComputerSkills,73462.0,13.428221,7.057835,0.0,2.0,4.0,8.0,13.0,17.0,26.00,35.00,107.0


### Logical Consistency Checks

In [37]:
logical_checks = {
    "negative_years_code": int((df.YearsCode < 0).sum()),
    "negative_years_code_pro": int((df.YearsCodePro < 0).sum()),
    "professional_experience_exceeds_total": int((df.YearsCodePro > df.YearsCode).sum()),
    "non_positive_salary": int((df.PreviousSalary <= 0).sum()),
    "negative_computer_skills": int((df.ComputerSkills < 0).sum()),
    "zero_computer_skills": int((df.ComputerSkills == 0).sum())
}

logical_checks

{'negative_years_code': 0,
 'negative_years_code_pro': 0,
 'professional_experience_exceeds_total': 588,
 'non_positive_salary': 0,
 'negative_computer_skills': 0,
 'zero_computer_skills': 63}

### Salary Distribution Check

In [38]:
salary_distribution = {
    "mean_salary": float(df.PreviousSalary.mean()),
    "median_salary": int(df.PreviousSalary.median()),
    "mean_median_difference": float((df.PreviousSalary.mean()) - (df.PreviousSalary.median())),
    "mean_median_ratio": float((df.PreviousSalary.mean()) / (df.PreviousSalary.median()))
}

salary_distribution

{'mean_salary': 67750.2606109281,
 'median_salary': 57588,
 'mean_median_difference': 10162.260610928104,
 'mean_median_ratio': 1.176464899127042}

### Salary Range Inspection

In [39]:
salary_threshold_checks = {
    "salary_below_1000": int((df.PreviousSalary < 1000).sum()),
    "salary_below_5000": int((df.PreviousSalary < 5000).sum()),
    "salary_above_200000": int((df.PreviousSalary > 200000).sum()),
}
salary_threshold_checks

{'salary_below_1000': 480,
 'salary_below_5000': 2517,
 'salary_above_200000': 808}

In [40]:
salary_profile_columns = ["Country", "EdLevel", "YearsCodePro", "PreviousSalary", "ComputerSkills"]

df[salary_profile_columns].nlargest(5, "PreviousSalary")

,Country,EdLevel,YearsCodePro,PreviousSalary,ComputerSkills
19039,United States of America,Undergraduate,25,224000.0,15
48016,Germany,Undergraduate,29,223952.0,24
33689,New Zealand,Undergraduate,8,223615.0,2
64306,United States of America,Master,30,223115.0,9
42923,United States of America,Master,40,223000.0,15


In [41]:
df[salary_profile_columns].nsmallest(5, "PreviousSalary")

,Country,EdLevel,YearsCodePro,PreviousSalary,ComputerSkills
988,United States of America,Master,25,1.0,16
11973,United States of America,Undergraduate,9,1.0,10
22799,United States of America,Undergraduate,12,1.0,6
50178,Belgium,NoHigherEd,50,1.0,37
58657,India,Master,11,1.0,21


### Experience Consistency Inspection

In [42]:
experience_checks = {
    "zero_years_code": int((df.YearsCode == 0).sum()),
    "zero_years_code_pro": int((df.YearsCodePro == 0).sum()),
    "equal_total_and_professional_experience": int((df.YearsCode == df.YearsCodePro).sum()),
    "professional_experience_exceeds_total": int((df.YearsCodePro > df.YearsCode).sum())
}

experience_checks

{'zero_years_code': 187,
 'zero_years_code_pro': 2943,
 'equal_total_and_professional_experience': 4538,
 'professional_experience_exceeds_total': 588}

In [43]:
invalid_experience_mask = df.YearsCodePro > df.YearsCode

experience_profile_columns = ["Country", "EdLevel", "YearsCode", "YearsCodePro", "PreviousSalary", "ComputerSkills"]

df.loc[invalid_experience_mask, experience_profile_columns].head(10)

,Country,EdLevel,YearsCode,YearsCodePro,PreviousSalary,ComputerSkills
55,United Kingdom of Great Britain and Northern I...,Undergraduate,1,3,38778.0,20
316,India,Undergraduate,6,7,31332.0,6
485,Brazil,Undergraduate,20,24,6324.0,9
708,France,Master,12,18,51887.0,8
712,Brazil,Other,5,8,5496.0,14
739,Colombia,Other,2,7,8124.0,7
1011,India,Undergraduate,4,5,10052.0,16
1256,India,Undergraduate,1,3,9075.0,3
1278,Spain,Other,3,5,19452.0,41
1424,India,Undergraduate,4,5,7051.0,19


### Numerical Feature Assessment

The numerical features contain no missing values and no negative values. `YearsCode` and `YearsCodePro` range from 0 to 50, which is plausible for developer experience data.

The salary distribution is right-skewed. The mean salary is approximately 67,750, while the median is 57,588. The mean is therefore approximately 17.6% higher than the median, indicating that high salary observations pull the average upward.

The lower end of the salary distribution requires further investigation. There are 480 observations below 1,000 and 2,517 observations below 5,000. The five lowest salaries are all equal to 1, including profiles with substantial professional experience. These values may represent missing, hidden, incorrectly transformed, or non-comparable salary information rather than genuine annual salaries. However, they will not be removed until the salary definition, unit, and country effects are investigated.

There are 808 salaries above 200,000. These observations are located near the upper end of the dataset but are not automatically invalid. Country, experience, and the original salary construction should be considered before defining an outlier policy.

The experience consistency checks identified 187 profiles with zero total coding experience and 2,943 profiles with zero professional coding experience. Zero professional experience is plausible for students, hobbyists, and developers who have not yet worked professionally.

A total of 4,538 profiles report equal total and professional coding experience. This is possible, although it may also reflect rounding or different interpretations of the survey questions.

There are 588 profiles where professional coding experience exceeds total coding experience. This is logically inconsistent under the assumed column definitions and represents approximately 0.8% of the dataset. These records should be flagged during preprocessing rather than corrected or deleted during the audit stage.

`ComputerSkills` has a median of 13 and a 99th percentile of 35, while its maximum is 107. This large gap suggests a possible upper outlier. The value should be validated against the number of semicolon-separated technologies in `HaveWorkedWith` before deciding whether it is erroneous.

No numerical rows have been modified during this audit. The findings will later be converted into explicit preprocessing rules that are fitted using training data only.

## Categorical Feature Audit

### Low-Cardinality Feature Distributions

In [44]:
low_cardinality_columns = ["Age", "Accessibility", "EdLevel", "Employment", "Gender", "MentalHealth", "MainBranch", "Employed"]

for column in low_cardinality_columns:
    category_counts = df[column].value_counts(dropna=False)
    
    category_percentages = (
        df[column]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )
    
    category_distribution = pd.concat(
        [category_counts, category_percentages],
        axis=1
    )
    
    category_distribution.columns = ["count", "percentage"]
    
    print(column)
    display(category_distribution)

Age


,count,percentage
Age,,
<35,47819,65.09
>35,25643,34.91


Accessibility


,count,percentage
Accessibility,,
No,71355,97.13
Yes,2107,2.87


EdLevel


...

,count,percentage
MentalHealth,,
No,56944,77.51
Yes,16518,22.49


MainBranch


,count,percentage
MainBranch,,
Dev,67396,91.74
NotDev,6066,8.26


Employed


,count,percentage
Employed,,
1,39392,53.62
0,34070,46.38


In [45]:
employment_crosstab = pd.crosstab(
    df["Employment"],
    df["Employed"],
    margins=True
)

employment_crosstab

Employed,0,1,All
Employment,,,
0,3840,4748,8588
1,30230,34644,64874
All,34070,39392,73462


In [46]:
employment_percentage_crosstab = (
    pd.crosstab(
        df["Employment"],
        df["Employed"],
        normalize="index"
    )
    .mul(100)
    .round(2)
)

employment_percentage_crosstab

Employed,0,1
Employment,,
0,44.71,55.29
1,46.60,53.40


In [47]:
employment_agreement_rate = float(round(
    (df["Employment"] == df["Employed"]).mean() * 100,
    2
))

employment_agreement_rate

52.39

### Employed Target Dependency Check

In [48]:
target_audit_numerical_columns = ["YearsCode", "YearsCodePro", "PreviousSalary", "ComputerSkills"]

numerical_profile_by_employed = (
    df.groupby("Employed")[target_audit_numerical_columns]
    .agg(["mean", "median"])
    .round(2)
)

numerical_profile_by_employed

YearsCode        YearsCodePro        PreviousSalary           \
              mean median         mean median           mean   median   
Employed                                                                
0            14.26   12.0         9.07    6.0       67730.07  57588.0   
1            14.19   12.0         9.12    7.0       67767.72  57588.0   

         ComputerSkills         
                   mean median  
Employed                        
0                  8.98    9.0  
1                 17.27   16.0

### Employment Rate by Skill Count

In [49]:
employed_rate_by_skill_count = (
    df.groupby("ComputerSkills")["Employed"]
    .agg(
        profile_count="size",
        employed_rate="mean"
    )
)

employed_rate_by_skill_count["employed_rate"] = (
    employed_rate_by_skill_count["employed_rate"]
    .mul(100)
    .round(2)
)

employed_rate_by_skill_count

,profile_count,employed_rate
ComputerSkills,,
0,63,0.00
1,370,0.00
2,1012,0.00
3,1557,0.39
4,2152,1.53
...,...,...
90,1,100.00
91,2,100.00
101,1,100.00


### Target Audit Decision

`Employment` and `Employed` do not represent the same information. Their agreement rate is only 52.39%, and the distribution of `Employed` is strongly associated with `ComputerSkills` rather than actual employment status. Therefore, `Employed` will not be used as an employment prediction target, and the project will not present its output as a job-finding probability.